# Verificación de las cifras del manuscrito

Este cuaderno existe para que cualquier persona pueda comprobar, sin conocer el proyecto y sin
correr el análisis completo, que las cifras publicadas en el manuscrito se obtienen de los datos.

Hace tres cosas, en este orden:

1. **Comprueba los insumos.** Verifica que estén los archivos intermedios que dejaron los
   cuadernos anteriores, y deja constancia de su tamaño y de su número de filas.
2. **Calcula el subregistro diferencial por aseguramiento.** Es el único resultado del manuscrito
   que no producía ningún otro cuaderno, porque exige la matriz completa anterior a las
   exclusiones y todos los demás trabajan sobre la muestra analítica.
3. **Contrasta cifra por cifra.** Recalcula cada valor declarado en el manuscrito a partir de los
   datos y de las salidas guardadas, y emite un cuadro de comprobación con el resultado de cada
   contraste.

## Cómo se ejecuta

Se ejecuta de principio a fin, sin ajustar nada. No reajusta ningún modelo: lee los intermedios
que dejaron los cuadernos `01` a `08` y las tablas de `salidas/tablas/`. En una máquina corriente
tarda del orden de dos minutos, salvo la sección 2, que recorre los dieciocho millones de
registros de la matriz completa y puede tardar algo más.

Quien reproduzca este análisis desde cero debe correr antes los cuadernos `01` a `08` en orden.

## Qué deja escrito

| Archivo | Contenido |
|---|---|
| `salidas/tablas/subregistro_aseguramiento.csv` y `.tex` | El contraste de la sección 2 |
| `salidas/tablas/verificacion_cifras.csv` y `.tex` | El cuadro de comprobación de la sección 3 |


## 1. Insumos

Antes de comprobar nada conviene dejar constancia de sobre qué se está comprobando. Un cuaderno de
verificación que no declara sus insumos no verifica: repite.

Se comprueban dos cosas distintas. Los **intermedios** son los `.parquet` y el `.pkl` que dejaron
los cuadernos anteriores, y sin ellos este cuaderno no puede hacer nada. La **matriz completa** es
la base de los 18.008.809 registros anteriores a las exclusiones del Cuadro 4.1, y solo la
necesita la sección 2.

La matriz completa no forma parte de la carpeta portátil, porque pesa más que todo lo demás junto.
Si no está disponible, la sección 2 se salta y lo declara en lugar de fallar en silencio: es
preferible un cuaderno que diga qué no pudo comprobar a uno que aparente haberlo comprobado todo.

In [ ]:
# Bibliotecas del análisis. Ninguna es opcional.
import sys                                    # para añadir el directorio del código al path
import time                                   # para cronometrar las celdas pesadas
from pathlib import Path                      # rutas independientes del sistema operativo

import numpy as np
import pandas as pd

# El directorio del código es el padre de notebooks/. Se añade al path para poder
# importar src/, que es donde vive el mapa de armonización que la sección 2 necesita.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

INTERMEDIOS = RAIZ / "intermedios"            # lo que dejaron los cuadernos 01 a 08
TABLAS      = RAIZ / "salidas" / "tablas"     # las salidas que el manuscrito cita

# La matriz completa vive fuera de la carpeta portátil. Se buscan las ubicaciones
# conocidas por orden; la primera que exista es la que se usa.
CANDIDATAS = [
    RAIZ / "datos" / "matriz_completa_1998_2024.parquet",
    Path.home() / "Desktop" / "TESIS" / "DATA" / "BPN_1998_2024" / "matriz_completa_1998_2024.parquet",
    Path(r"C:\Users\ASUS\Desktop\TESIS\DATA\BPN_1998_2024\matriz_completa_1998_2024.parquet"),
]
MATRIZ = next((p for p in CANDIDATAS if p.exists()), None)

# Los intermedios que este cuaderno necesita, con las filas que debe tener cada uno.
# El valor None significa que no procede comprobar el número de filas.
ESPERADOS = {
    "muestra_analitica.parquet":    17_376_860,
    "prevalencia_municipio.parquet": None,
    "modelos_multinivel.pkl":        None,
}

print("INSUMOS\n" + "=" * 78)
faltan = []                                   # se acumulan los que no estén, para el veredicto
for nombre, filas_esperadas in ESPERADOS.items():
    ruta = INTERMEDIOS / nombre
    if not ruta.exists():                     # si falta, se anota y se sigue
        print(f"  FALTA        {nombre}")
        faltan.append(nombre)
        continue
    mb = ruta.stat().st_size / 1024**2        # tamaño en megabytes, para dejar constancia
    if ruta.suffix == ".parquet":
        import pyarrow.parquet as pq          # se lee solo el pie del archivo, no los datos
        filas = pq.ParquetFile(ruta).metadata.num_rows
        marca = "" if filas_esperadas is None else ("  OK" if filas == filas_esperadas else "  DISCREPA")
        print(f"  {nombre:<32} {filas:>12,} filas  {mb:>8.1f} MB{marca}")
    else:
        print(f"  {nombre:<32} {'':>12}         {mb:>8.1f} MB")

# La muestra analitica pesa 232 MB y no viaja en el repositorio: la regeneran los
# cuadernos 01 y 01b. Sin ella, las secciones que la recorren se saltan y sus
# comprobaciones quedan marcadas como no evaluadas, en vez de interrumpir el cuaderno.
HAY_MUESTRA = ((INTERMEDIOS / "muestra_analitica.parquet").exists()
               or (INTERMEDIOS / "muestra_analitica").is_dir())

print(f"\n  tablas de salida disponibles: {len(list(TABLAS.glob('*.csv')))}")
print(f"  muestra analitica: {'disponible' if HAY_MUESTRA else 'NO DISPONIBLE (se saltan las secciones que la recorren)'}")
print(f"  matriz completa: {'encontrada' if MATRIZ else 'NO DISPONIBLE (la sección 2 se saltará)'}")
if MATRIZ:
    print(f"    en {MATRIZ}")

### Lectura

La muestra analítica debe tener exactamente 17.376.860 filas, que es la cifra del Cuadro 4.1 y la
que el Capítulo 5 declara como base de todos los cálculos. Una discrepancia aquí invalidaría todo
lo que sigue, porque significaría que los intermedios no corresponden a la versión del manuscrito
que se está comprobando.

El resto de los insumos se declara sin comprobación de tamaño: se usan para leer parámetros ya
estimados, no para recalcularlos.

Si la matriz completa aparece como no disponible, las cuatro comprobaciones del subregistro
quedarán marcadas como no evaluadas en el cuadro final. El resto del cuaderno funciona igual.

## 2. El subregistro diferencial por aseguramiento

Esta sección calcula el contraste que respalda la tercera limitación del Capítulo 6, y responde a
la objeción de que el resultado del régimen de afiliación proceda de quién queda registrado y no
de quién tiene mayor riesgo.

La hipótesis es que una mujer sin aseguramiento tiene menor probabilidad de que su parto quede
registrado **con el peso informado**, de modo que el grupo observado no sería el grupo real sino
el de quienes llegaron al sistema. Si eso ocurre, la desigualdad estimada es una cota inferior de
la real.

**Por qué se calcula sobre la matriz completa y no sobre la muestra analítica.** La muestra
analítica excluye precisamente los registros sin peso informado, que son el objeto de esta
medición. Medir la ausencia sobre la base que ya la eliminó daría cero por construcción. Por eso
esta sección es la única del proyecto que trabaja sobre los 18.008.809 registros previos a las
exclusiones del Cuadro 4.1.

**Cómo se identifica la ausencia.** `PESO_NAC` no viene en gramos sino en ocho bandas de 500
gramos, y el código 9 significa «sin información». Se cuentan como ausentes ese código y los
valores nulos.

**Cómo se armoniza el régimen.** Los códigos 3, 4 y 5 de `SEG_SOCIAL` cambian de significado en
2008, de modo que contarlos sin armonizar mezclaría categorías distintas. Se usa el mismo mapa
`SEG4` de `src/armonizar.py` que emplean los modelos, y no una recodificación propia, para que
esta sección y el Capítulo 5 hablen de los mismos grupos.

In [ ]:
if MATRIZ is None:
    print("Matriz completa no disponible: esta sección no se ejecuta.")
    subregistro = None                        # el cuadro de la sección 3 lo tratará como no evaluado
else:
    t0 = time.time()
    from src import armonizar as arm          # el mapa de armonización que usan los modelos

    # Se leen solo las tres columnas necesarias. Leer las 53 sobre 18 millones de filas
    # consume varios gigabytes sin ninguna ganancia.
    bruto = pd.read_parquet(MATRIZ, columns=["ANIO", "PESO_NAC", "SEG_SOCIAL"])

    codigo = pd.to_numeric(bruto["SEG_SOCIAL"], errors="coerce")   # el código tal como viene
    anio   = bruto["ANIO"].astype(int)

    # El mapa SEG4 tiene una tabla por régimen de codificación. Se aplica cada una
    # al subconjunto de años que le corresponde: antes y desde el año de corte.
    esp  = arm.ESCALAS["SEG4"]
    seg4 = pd.Series(np.nan, index=bruto.index)
    for regimen, mapa in esp["mapa"].items():
        filas = anio < arm.ANIO_CORTE if regimen == arm.REG_ANTIGUO else anio >= arm.ANIO_CORTE
        seg4.loc[filas] = codigo[filas].map(mapa)

    # Ausencia del peso: el código 9 del DANE, y los nulos.
    peso     = pd.to_numeric(bruto["PESO_NAC"], errors="coerce")
    sin_peso = peso.isna() | (peso == 9)

    # El código 0 no existe en la escala: se usa aquí para agrupar los registros cuyo
    # código de régimen no es reducible a las cuatro categorías comunes, que se declaran
    # aparte en vez de repartirse entre ellas.
    marco = pd.DataFrame({"seg4": seg4.fillna(0), "sin_peso": sin_peso})
    subregistro = (marco.groupby("seg4")
                        .agg(registros=("sin_peso", "size"), sin_peso=("sin_peso", "sum")))
    subregistro["pct_sin_peso"] = 100 * subregistro["sin_peso"] / subregistro["registros"]

    # La razón se expresa contra el régimen contributivo, que es el de mejor completitud
    # y el que sirve de referencia en el Capítulo 5.
    base = subregistro.loc[1.0, "pct_sin_peso"]
    subregistro["razon_vs_contributivo"] = subregistro["pct_sin_peso"] / base

    nombres = dict(esp["niveles"]); nombres[0] = "Sin régimen mapeable"
    subregistro.insert(0, "regimen", [nombres[int(i)] for i in subregistro.index])
    subregistro = subregistro.sort_values("pct_sin_peso", ascending=False).reset_index(drop=True)

    print(subregistro.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))
    print(f"\n  total sin peso informado: {int(sin_peso.sum()):,}")
    print(f"  tiempo: {time.time() - t0:.1f} s")

### Lectura

La ausencia del peso al nacer es marcadamente diferencial por régimen de afiliación. Entre las
mujeres no aseguradas alcanza el 2,943 % de los registros, frente al 0,214 % entre las afiliadas
al régimen contributivo: una razón de 13,8. El total de registros sin peso informado, 332.369,
coincide con el renglón correspondiente del Cuadro 4.1, lo que confirma que esta sección mide
exactamente lo que el trabajo excluye.

Los 620.472 registros sin régimen mapeable presentan la ausencia más alta de todas, el 21,9 %.
Esa coincidencia es esperable y refuerza la lectura: se trata mayoritariamente de certificados
incompletos, donde falta tanto el régimen como el peso.

El sentido de este resultado es el que importa. Si quienes quedan fuera del registro son
sistemáticamente las de menor posición socioeconómica, el efecto social estimado sobre las que sí
quedaron registradas **subestima** el real. El sesgo por ausencia diferencial opera en contra de
la hipótesis del trabajo y no a su favor.

### Evolución del contraste a lo largo del período

La cifra agregada del período puede ocultar dos historias distintas: que la brecha se haya
mantenido, o que se haya movido. El Capítulo 5 afirma que se amplía, y conviene comprobarlo por
separado porque de ello depende cómo se lee la aceleración final de la prevalencia.

Los tres tramos corresponden a los dos regímenes de codificación del certificado y al período
posterior a 2018, que es donde la serie de desigualdad cambia de signo.

In [ ]:
if MATRIZ is None:
    print("Matriz completa no disponible: esta sección no se ejecuta.")
    evolucion = None
else:
    filas = []
    for desde, hasta in [(1998, 2007), (2008, 2018), (2019, 2024)]:
        tramo = marco[(anio >= desde) & (anio <= hasta)]          # el subconjunto de años
        g = (tramo[tramo["seg4"] > 0]                              # se excluye el no mapeable
                  .groupby("seg4")
                  .agg(registros=("sin_peso", "size"), sin_peso=("sin_peso", "sum")))
        g["pct"] = 100 * g["sin_peso"] / g["registros"]
        filas.append({
            "tramo":            f"{desde}--{hasta}",
            "pct_no_asegurada": g.loc[4.0, "pct"],                 # categoría 4 = no asegurada
            "pct_contributivo": g.loc[1.0, "pct"],                 # categoría 1 = contributivo
            "razon":            g.loc[4.0, "pct"] / g.loc[1.0, "pct"],
        })
    evolucion = pd.DataFrame(filas)
    print(evolucion.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))

### Lectura

La razón pasa de 7,4 en 1998--2007 a 32,0 en 2019--2024, es decir, se multiplica por más de
cuatro. Y lo hace por un motivo que conviene enunciar con precisión, porque el sentido no es
obvio: la completitud del grupo no asegurado **mejoró** en términos absolutos, del 3,54 % de
ausencia al 1,49 %. Lo que ocurre es que la del régimen contributivo mejoró mucho más, del 0,48 %
al 0,05 %.

La brecha se amplía, por tanto, sobre un fondo de mejora general del registro. Esa distinción
importa para la lectura del Capítulo 6: el subregistro diferencial no es un deterioro reciente del
sistema de información, sino una mejora que no alcanzó por igual a todos los grupos.

### Persistencia de la salida

Las dos tablas anteriores se guardan en `salidas/tablas/`, en formato `.csv` para su consulta y
`.tex` para su inserción directa en el manuscrito. El `.tex` se emite con el entorno `tabular`
completo y no solo con sus filas, porque en la instalación de LaTeX empleada un `\input` que
contenga únicamente filas dentro de un `tabular` produce un error de `\noalign`.

In [ ]:
if subregistro is not None:
    TABLAS.mkdir(parents=True, exist_ok=True)
    subregistro.to_csv(TABLAS / "subregistro_aseguramiento.csv", index=False)
    evolucion.to_csv(TABLAS / "subregistro_evolucion.csv", index=False)

    def tex_numero(x, decimales=3):
        """Formatea un número al estilo del manuscrito: coma decimal protegida."""
        return "$" + f"{x:,.{decimales}f}".replace(",", "@").replace(".", "{,}").replace("@", ".") + "$"

    lineas = [r"\begin{tabular}{lrrrr}", r"\toprule",
              r"\textbf{Régimen} & \textbf{Registros} & \textbf{Sin peso} & "
              r"\textbf{\% sin peso} & \textbf{Razón} \\", r"\midrule"]
    for _, f in subregistro.iterrows():
        lineas.append(f"{f['regimen']} & {tex_numero(f['registros'], 0)} & "
                      f"{tex_numero(f['sin_peso'], 0)} & {tex_numero(f['pct_sin_peso'])} & "
                      f"{tex_numero(f['razon_vs_contributivo'], 1)} \\\\")
    lineas += [r"\bottomrule", r"\end{tabular}"]
    (TABLAS / "subregistro_aseguramiento.tex").write_text("\n".join(lineas), encoding="utf-8")

    print("escritas:")
    for n in ["subregistro_aseguramiento.csv", "subregistro_evolucion.csv",
              "subregistro_aseguramiento.tex"]:
        print(f"  salidas/tablas/{n}")

## 3. La banda de 28 a 37 semanas, descompuesta

Esta sección produce el Cuadro~5.21 del manuscrito y las cifras que lo acompañan.

El certificado consigna **dos** clasificaciones de la duración de la gestación. `T_GES` cubre los
veintisiete años y su tercera banda es «de 28 a 37 semanas», de modo que el corte de 37 semanas de
la Organización Mundial de la Salud cae dentro de ella. `T_GES_AGRU_CIE` sigue el corte de la
Organización, con una tercera banda «de 28 a 36» y una cuarta «de 37 a 41», pero solo existe en
catorce de los veintisiete años: 2008 a 2011, 2014 a 2019 y 2021 a 2024.

Cruzar las dos permite establecer qué parte de la banda gruesa es prematuridad y qué parte son
nacimientos de la semana 37.

**Sobre qué se calcula.** Todas las cantidades de esta sección son conteos, de modo que se obtienen
de una tabla de celdas y no de los registros individuales. `celdas_gestacion.parquet` agrega la
muestra analítica por año, banda de `T_GES`, banda de la clasificación internacional y tipo de
parto: 586 filas que suman los 17.376.860 nacimientos. Pesa 7 kB y viaja en el repositorio.

Ese es el motivo de trabajar sobre él y no sobre la muestra: las cifras son idénticas, porque la
agregación es exacta, y así esta sección puede ejecutarse sin descargar los 232 MB de la muestra
analítica. Si la muestra está disponible, la primera celda regenera la tabla de celdas antes de
usarla; si no lo está, emplea la que viene en el repositorio.

In [ ]:
CELDAS = INTERMEDIOS / "celdas_gestacion.parquet"

if HAY_MUESTRA:
    # La muestra esta: se regenera la tabla de celdas, de modo que nunca quede
    # desactualizada respecto de los datos.
    t0 = time.time()
    _m = INTERMEDIOS / "muestra_analitica.parquet"
    if not _m.exists():                 # en el repositorio viaja particionada por anio
        _m = INTERMEDIOS / "muestra_analitica"
    bruto = pd.read_parquet(_m,
                            columns=["ANIO", "T_GES", "T_GES_AGRU_CIE", "TIPO_PARTO", "BPN"])
    # Los codigos se pasan a numero: vienen como categoricas con dominios distintos
    # y pandas no permite compararlas entre si.
    marco = pd.DataFrame({
        "ANIO":       bruto["ANIO"].to_numpy(dtype="int16"),
        "T_GES":      pd.to_numeric(bruto["T_GES"], errors="coerce").to_numpy(dtype="float32"),
        "CIE":        pd.to_numeric(bruto["T_GES_AGRU_CIE"], errors="coerce").to_numpy(dtype="float32"),
        "TIPO_PARTO": pd.to_numeric(bruto["TIPO_PARTO"], errors="coerce").to_numpy(dtype="float32"),
        "BPN":        bruto["BPN"].to_numpy(dtype="int8"),
    })
    del bruto
    celdas = (marco.groupby(["ANIO", "T_GES", "CIE", "TIPO_PARTO"], dropna=False)
                   .agg(n=("BPN", "size"), casos=("BPN", "sum")).reset_index())
    celdas["n"] = celdas["n"].astype("int32")
    celdas["casos"] = celdas["casos"].astype("int32")
    del marco
    celdas.to_parquet(CELDAS, index=False, compression="zstd")
    print(f"tabla de celdas regenerada desde la muestra en {time.time() - t0:.1f} s")
else:
    celdas = pd.read_parquet(CELDAS)
    print("tabla de celdas leida del repositorio (la muestra analitica no esta)")

print(f"  {len(celdas)} celdas  |  {celdas['n'].sum():,} nacimientos  |  "
      f"{celdas['casos'].sum():,} de bajo peso")

# Comprobacion de integridad: la agregacion debe conservar el total de la muestra.
# Si no lo hace, la tabla no corresponde a esta version de los datos.
assert int(celdas["n"].sum()) == 17_376_860, "la tabla de celdas no suma la muestra analitica"
print("  la tabla suma exactamente la muestra analitica")

### Lectura

La tabla de celdas debe sumar los 17.376.860 nacimientos de la muestra analítica. La comprobación
es una aserción y no un aviso: si no se cumple, el cuaderno se detiene, porque todas las cifras de
esta sección y de la siguiente se calculan sobre ella.

Las 586 celdas son el producto de los veintisiete años por las bandas de las dos clasificaciones y
los tipos de parto, descontadas las combinaciones que no ocurren.

### El reparto de la banda

Se restringe a los años en que existe la segunda clasificación, y se comprueba primero que las dos
coinciden fuera de la banda 3. Esa comprobación no es formal: si discreparan también en las demás
bandas, no serían la misma información con distinto agrupamiento y la descomposición que sigue no
tendría sentido.

In [ ]:
cruce = celdas[celdas["CIE"].notna()]

# Concordancia fuera de la banda 3, ponderada por el numero de nacimientos de cada celda.
fuera = cruce[cruce["T_GES"] != 3]
concuerdan = fuera[fuera["T_GES"] == fuera["CIE"]]["n"].sum()
print(f"concordancia fuera de la banda 3: {100 * concuerdan / fuera['n'].sum():.2f} %")

filas = []
for anio in [2008, 2014, 2019, 2024]:
    x = cruce[cruce["ANIO"] == anio]
    total_anio = x["n"].sum()
    b3 = x[x["T_GES"] == 3]
    pretermino = b3[b3["CIE"] == 3]["n"].sum()    # 28 a 36: pretermino verdadero
    semana37   = b3[b3["CIE"] == 4]["n"].sum()    # 37: ya a termino
    filas.append({
        "anio":        anio,
        "banda_28_37": 100 * b3["n"].sum() / total_anio,
        "pretermino":  100 * pretermino / total_anio,
        "semana_37":   100 * semana37 / total_anio,
        "pct_sem37":   100 * semana37 / b3["n"].sum(),
    })
banda3 = pd.DataFrame(filas)
print()
print(banda3.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

subida_pre = banda3.loc[3, "pretermino"] - banda3.loc[0, "pretermino"]
subida_s37 = banda3.loc[3, "semana_37"]  - banda3.loc[0, "semana_37"]
total_subida = subida_pre + subida_s37
print(f"\nCRECIMIENTO 2008 a 2024, en puntos porcentuales")
print(f"  pretermino verdadero (28 a 36) ... {subida_pre:+6.2f}   "
      f"({100 * subida_pre / total_subida:4.0f} %)")
print(f"  semana 37 ....................... {subida_s37:+6.2f}   "
      f"({100 * subida_s37 / total_subida:4.0f} %)")
print(f"  total de la banda ............... {total_subida:+6.2f}")

### Lectura

Las dos clasificaciones coinciden en la práctica totalidad de los nacimientos fuera de la banda 3,
lo que confirma que se trata de la misma información con distinto agrupamiento. La discrepancia se
concentra en esa banda, y consiste en que la clasificación de la Organización reasigna a los
nacimientos a término los de la semana 37.

Entre 2008 y 2024 la banda crece 9,07 puntos porcentuales, y el 79 % de ese crecimiento son
nacimientos de la semana 37. La prematuridad verdadera también aumenta, pero se multiplica por
1,22 frente al 1,50 de la banda completa.

El desplazamiento entre bandas, que el índice agregado recoge como si fuera un aumento de la
prematuridad, es en sus cuatro quintas partes un desplazamiento hacia el parto en la semana 37.

### Prevalencia por subgrupo y tipo de parto

Las dos cantidades que siguen sostienen sendas afirmaciones del Capítulo 5.

La prevalencia dentro de cada subgrupo justifica que el grupo a término del resultado principal se
defina como 38 a 41 y no como 37 a 41: si la semana 37 se incorporase al grupo de referencia, su
prevalencia se multiplicaría por ocho y el grupo dejaría de ser aquel en que el bajo peso solo
puede atribuirse a restricción del crecimiento.

La proporción de cesáreas acompaña al crecimiento de la semana 37, que es la primera en que un
embarazo se considera a término y, por tanto, el momento en que se programa una intervención. La
estandarización directa cuantifica hasta dónde llega esa explicación.

In [ ]:
x24 = cruce[cruce["ANIO"] == 2024]
b3_24 = x24[x24["T_GES"] == 3]

print("PREVALENCIA DE BAJO PESO POR SUBGRUPO, 2024")
subgrupos = {
    "28 a 36, pretérmino": b3_24[b3_24["CIE"] == 3],
    "semana 37":           b3_24[b3_24["CIE"] == 4],
    "38 a 41, a término":  x24[x24["T_GES"] == 4],
}
prev_subgrupo = {}
for etiqueta, sub in subgrupos.items():
    prev_subgrupo[etiqueta] = 100 * sub["casos"].sum() / sub["n"].sum()
    print(f"  {etiqueta:<22} n = {sub['n'].sum():>8,}   "
          f"bajo peso = {prev_subgrupo[etiqueta]:>6.2f} %")

# Cesarea sobre la serie completa. El codigo 2 de TIPO_PARTO es la cesarea.
partos = celdas[celdas["TIPO_PARTO"].notna()]
por_anio = partos.groupby("ANIO")["n"].sum()
cesareas = partos[partos["TIPO_PARTO"] == 2].groupby("ANIO")["n"].sum()
cesarea = 100 * cesareas / por_anio
print(f"\nCESÁREA: {cesarea.loc[1998]:.1f} % en 1998  ->  {cesarea.loc[2024]:.1f} % en 2024")

# Estandarizacion directa: se fija la composicion por tipo de parto en la de 1998 y
# se recalcula la banda corta. La diferencia con la serie cruda es la parte del
# crecimiento atribuible al cambio de composicion entre parto espontaneo y cesarea.
w = partos[partos["TIPO_PARTO"].isin([1.0, 2.0])].copy()
w["cortos"] = np.where(w["T_GES"] == 3, w["n"], 0)

ref = w[w["ANIO"] == 1998].groupby("TIPO_PARTO")["n"].sum()
ref = ref / ref.sum()                                  # composicion de 1998
tasas = (w.groupby(["ANIO", "TIPO_PARTO"])
          .apply(lambda d: d["cortos"].sum() / d["n"].sum(), include_groups=False)
          .unstack())
estandarizada = (tasas * ref).sum(axis=1) * 100        # tasas de cada anio, pesos de 1998
cruda = (w.groupby("ANIO")
          .apply(lambda d: d["cortos"].sum() / d["n"].sum(), include_groups=False)) * 100

sub_cruda = cruda.loc[2024] - cruda.loc[1998]
sub_est   = estandarizada.loc[2024] - estandarizada.loc[1998]
pct_tipo_parto = 100 * (sub_cruda - sub_est) / sub_cruda
print(f"\nESTANDARIZACIÓN DIRECTA POR TIPO DE PARTO, 1998 a 2024")
print(f"  crecimiento observado ................... {sub_cruda:6.2f} puntos")
print(f"  crecimiento con composición de 1998 ..... {sub_est:6.2f} puntos")
print(f"  atribuible al cambio de tipo de parto ... {pct_tipo_parto:6.1f} %")

### Lectura

La prevalencia de bajo peso es del 58,07 % entre los nacimientos de 28 a 36 semanas, del 17,90 % en
los de la semana 37 y del 2,21 % en los de 38 a 41. El grupo de la semana 37 es intermedio en
riesgo y más numeroso que el pretérmino, de modo que su inclusión o exclusión desplaza
apreciablemente cualquier cantidad calculada sobre la banda.

La cesárea pasa del 26,2 % de los nacimientos en 1998 al 52,0 % en 2024. La estandarización
directa atribuye al cambio de composición entre parto espontáneo y cesárea el 17,8 % del
crecimiento de la banda corta: el resto ocurre dentro de cada tipo de parto, que también se
desplaza. La cesárea acompaña al fenómeno y no lo agota, y el manuscrito lo declara así.

## 4. Comprobación de las cifras del manuscrito

Esta sección es la razón de ser del cuaderno. Recalcula, a partir de los datos y de los parámetros
ya estimados, cada cifra que el manuscrito declara, y compara el valor obtenido con el publicado.

**Cómo se declara cada comprobación.** Cada una lleva la sección del manuscrito donde aparece la
cifra, el valor publicado, el valor recalculado aquí y la tolerancia admitida. La tolerancia no es
arbitraria: recoge el redondeo con que la cifra se imprime en el documento. Una cifra publicada
con dos decimales no puede exigirse con cuatro.

**Qué significa cada resultado.** `COINCIDE` significa que el valor recalculado cae dentro de la
tolerancia del publicado. `DISCREPA` significa que no, y obliga a revisar cuál de los dos está
desactualizado antes de dar la versión por buena. `NO EVALUADA` significa que el insumo necesario
no estaba disponible, y no debe leerse como conformidad.

In [ ]:
import pickle
from src import multinivel as mn

comprobaciones = []          # cada elemento: (id, sección, descripción, publicado, obtenido, tolerancia)

def comprobar(ident, seccion, descripcion, publicado, obtenido, tolerancia):
    """Registra una comprobación. `obtenido` en None marca la cifra como no evaluada."""
    comprobaciones.append({
        "id": ident, "seccion": seccion, "cifra": descripcion,
        "publicado": publicado, "obtenido": obtenido, "tolerancia": tolerancia,
    })

# --- Muestra -------------------------------------------------------------------
import pyarrow.parquet as pq
# El total de la tabla de celdas es, por construccion, el de la muestra analitica,
# y la asercion de la seccion 3 ya lo comprobo.
n_analitica = int(celdas["n"].sum())
comprobar("M1", "Cuadro 4.1", "Registros de la muestra analítica",
          17_376_860, n_analitica, 0)
comprobar("M2", "Cuadro 4.1", "Retención sobre los 18.008.809 registros (%)",
          96.49, 100 * n_analitica / 18_008_809, 0.005)

# --- Prevalencia ---------------------------------------------------------------
# Se leen solo las dos columnas necesarias: el año y el desenlace ya construido.
# BPN viene construido en la matriz y no se recalcula a partir de un umbral de gramos,
# porque PESO_NAC está bandeado en intervalos de 500 g.
# Las prevalencias son cocientes de conteos, de modo que salen exactas de la tabla
# de celdas de la seccion 3 y no exigen recorrer los 17 millones de registros.
anual = celdas.groupby("ANIO")[["n", "casos"]].sum()
comprobar("P1", "§5.1", "Prevalencia del período (%)",
          8.78, 100 * celdas["casos"].sum() / celdas["n"].sum(), 0.005)
comprobar("P2", "§5.1", "Prevalencia en 1998 (%)",
          7.64, 100 * anual.loc[1998, "casos"] / anual.loc[1998, "n"], 0.005)
comprobar("P3", "§5.1", "Prevalencia en 2024 (%)",
          11.15, 100 * anual.loc[2024, "casos"] / anual.loc[2024, "n"], 0.005)

# --- Componentes de varianza del modelo multinivel -----------------------------
# El archivo debe contener los ocho ajustes que deja el cuaderno 04. Un archivo con
# menos procede de una corrida que no terminó, y en ese caso las comprobaciones que
# dependen de los ajustes ausentes se registran como no evaluadas en vez de
# interrumpir el cuaderno: un fallo silencioso sería peor que una casilla vacía.
modelos = pickle.load(open(INTERMEDIOS / "modelos_multinivel.pkl", "rb"))
ESPERADOS_MODELOS = ["M0 vacio", "M1 social", "M2 ajustado", "M1c (submuestra de M2)",
                     "M3 +etnia", "M2b (submuestra de M3)", "M4 +mediador",
                     "M3b (submuestra de M4)"]
ausentes = [m for m in ESPERADOS_MODELOS if m not in modelos]
if ausentes:
    print("AVISO: faltan ajustes en modelos_multinivel.pkl ->", ", ".join(ausentes))
    print("       Ese archivo procede de una corrida incompleta. El árbol de código bueno")
    print("       es BPN_PORTATIL, que trae los ocho ajustes de la corrida completa.")
    print()

def del_modelo(nombre, funcion):
    """Devuelve el valor calculado sobre ese ajuste, o None si el ajuste no está."""
    return funcion(modelos[nombre]) if nombre in modelos else None

icc_m0 = mn.icc(modelos["M0 vacio"]) if "M0 vacio" in modelos else None

comprobar("V1", "§5.3", "Correlación intraclase municipal, M0 (%)",
          1.79, None if icc_m0 is None else 100 * icc_m0["icc_grupo"], 0.005)
comprobar("V2", "§5.3", "Correlación intraclase departamental, M0 (%)",
          0.85, None if icc_m0 is None else 100 * icc_m0["icc_supergrupo"], 0.005)
comprobar("V5", "§5.3", "Razón de momios mediana entre departamentos, M0",
          1.175, del_modelo("M0 vacio", lambda m: mn.mor(m["sigma_supergrupo"])), 0.0005)

# El Capítulo 5 afirma que algo más de la mitad de la variación territorial se sitúa
# entre municipios de un mismo departamento. Esa fracción es la varianza municipal
# sobre la suma de las dos varianzas territoriales, y conviene vigilarla porque se
# enunció durante una versión como «dos tercios», que no es lo que dan los datos.
comprobar("V6", "§5.3", "Fracción de la variación territorial que es municipal (%)",
          52.3, del_modelo("M0 vacio",
                lambda m: 100 * m["sigma_grupo"] ** 2
                          / (m["sigma_grupo"] ** 2 + m["sigma_supergrupo"] ** 2)), 0.5)
comprobar("V3", "§5.3", "Razón de momios mediana entre municipios, M0",
          1.184, del_modelo("M0 vacio", lambda m: mn.mor(m["sigma_grupo"])), 0.0005)
comprobar("V4", "§5.3", "Razón de momios mediana entre municipios, M2",
          1.197, del_modelo("M2 ajustado", lambda m: mn.mor(m["sigma_grupo"])), 0.0005)

# --- Descomposicion de la banda de gestacion ------------------------------------
# Todas proceden de la seccion 3 de este cuaderno, y sostienen el Cuadro 5.21 y el
# parrafo del Capitulo 5 que justifica el corte de 38 semanas.
# Estas once proceden de la seccion 3, y se calculan sobre la tabla de celdas, que
# siempre esta disponible.
comprobar("G1", "Cuadro 5.21", "Banda de 28 a 37 semanas en 2024 (%)",
          27.32, banda3.loc[3, "banda_28_37"], 0.005)
comprobar("G2", "Cuadro 5.21", "Pretermino verdadero (28 a 36) en 2024 (%)",
          10.65, banda3.loc[3, "pretermino"], 0.005)
comprobar("G3", "Cuadro 5.21", "Nacimientos de la semana 37 en 2024 (%)",
          16.67, banda3.loc[3, "semana_37"], 0.005)
comprobar("G4", "Cuadro 5.21", "Semana 37 como % de la banda, 2024",
          61.0, banda3.loc[3, "pct_sem37"], 0.05)
comprobar("G5", "Cuadro 5.21", "Parte del crecimiento que es semana 37 (%)",
          79.0, 100 * subida_s37 / total_subida, 0.5)
comprobar("G6", "§5.8.6", "Bajo peso entre 28 y 36 semanas, 2024 (%)",
          58.07, prev_subgrupo["28 a 36, pretérmino"], 0.005)
comprobar("G7", "§5.8.6", "Bajo peso en la semana 37, 2024 (%)",
          17.90, prev_subgrupo["semana 37"], 0.005)
comprobar("G8", "§5.8.6", "Bajo peso entre 38 y 41 semanas, 2024 (%)",
          2.21, prev_subgrupo["38 a 41, a término"], 0.005)
comprobar("G9",  "§5.8.6", "Cesarea en 1998 (%)", 26.2, cesarea.loc[1998], 0.05)
comprobar("G10", "§5.8.6", "Cesarea en 2024 (%)", 52.0, cesarea.loc[2024], 0.05)
comprobar("G11", "§5.8.6", "Crecimiento atribuible al tipo de parto (%)",
          17.8, pct_tipo_parto, 0.05)

# --- Subregistro ---------------------------------------------------------------
# Si la sección 2 no se ejecutó, se registran como no evaluadas en lugar de omitirlas.
if subregistro is not None:
    fila_no_aseg = subregistro.loc[subregistro["regimen"] == "No asegurado"].iloc[0]
    fila_contrib = subregistro.loc[subregistro["regimen"] == "Contributivo"].iloc[0]
    comprobar("S1", "§6.10", "Sin peso informado entre no aseguradas (%)",
              2.943, fila_no_aseg["pct_sin_peso"], 0.0005)
    comprobar("S2", "§6.10", "Sin peso informado en el contributivo (%)",
              0.214, fila_contrib["pct_sin_peso"], 0.0005)
    comprobar("S3", "§6.10", "Razón entre ambas proporciones",
              13.8, fila_no_aseg["razon_vs_contributivo"], 0.05)
    comprobar("S4", "§6.10", "Razón en 1998--2007", 7.4, evolucion.loc[0, "razon"], 0.05)
    comprobar("S5", "§6.10", "Razón en 2019--2024", 32.0, evolucion.loc[2, "razon"], 0.05)
else:
    for ident, desc, pub in [("S1", "Sin peso informado entre no aseguradas (%)", 2.943),
                             ("S2", "Sin peso informado en el contributivo (%)", 0.214),
                             ("S3", "Razón entre ambas proporciones", 13.8),
                             ("S4", "Razón en 1998--2007", 7.4),
                             ("S5", "Razón en 2019--2024", 32.0)]:
        comprobar(ident, "§6.10", desc, pub, None, 0)

print(f"comprobaciones registradas: {len(comprobaciones)}")

### El cuadro de comprobación

Se evalúa cada comprobación registrada y se emite el resultado. El cuadro es la salida que
acredita la reproducibilidad del manuscrito, y por eso se guarda también en disco: quien revise el
trabajo puede leerlo sin ejecutar nada.

In [ ]:
resultados = pd.DataFrame(comprobaciones)

def veredicto(f):
    """COINCIDE, DISCREPA o NO EVALUADA, según el valor obtenido y la tolerancia."""
    if f["obtenido"] is None or (isinstance(f["obtenido"], float) and np.isnan(f["obtenido"])):
        return "NO EVALUADA"
    return "COINCIDE" if abs(f["obtenido"] - f["publicado"]) <= f["tolerancia"] else "DISCREPA"

resultados["diferencia"] = resultados.apply(
    lambda f: np.nan if f["obtenido"] is None else f["obtenido"] - f["publicado"], axis=1)
resultados["resultado"] = resultados.apply(veredicto, axis=1)

# Se imprime con anchura fija para que el cuadro se lea en cualquier terminal.
def formatear(x):
    """Cuatro decimales para las proporciones; sin decimales para los conteos grandes."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "—"
    return f"{x:,.0f}" if abs(x) >= 10_000 else f"{x:,.4f}"

print(f"{'id':<4}{'sección':<12}{'cifra':<50}{'publicado':>14}{'obtenido':>14}   resultado")
print("=" * 108)
for _, f in resultados.iterrows():
    print(f"{f['id']:<4}{f['seccion']:<12}{f['cifra'][:48]:<50}"
          f"{formatear(f['publicado']):>14}{formatear(f['obtenido']):>14}   {f['resultado']}")

TABLAS.mkdir(parents=True, exist_ok=True)
resultados.to_csv(TABLAS / "verificacion_cifras.csv", index=False)
print(f"\nescrita: salidas/tablas/verificacion_cifras.csv")

### Lectura

Todas las cifras deben aparecer como `COINCIDE`. Una sola discrepancia obliga a detenerse: no
significa necesariamente que el manuscrito esté mal, sino que el manuscrito y los intermedios no
corresponden a la misma corrida, y hay que averiguar cuál de los dos está desactualizado antes de
dar la versión por entregable.

Las comprobaciones marcadas como `NO EVALUADA` no son conformidad. Indican que el insumo necesario
no estaba disponible en esta ejecución, y la cifra correspondiente sigue sin verificar.

**Sobre las dos correlaciones intraclase.** Ambas se calculan con el mismo denominador, la suma de
las tres componentes de varianza, y esa es la razón de comprobarlas juntas. La municipal lleva en
el numerador la suma de la varianza municipal y la departamental, porque compartir municipio
implica compartir departamento; la departamental lleva únicamente la departamental. Emplear para
la segunda un denominador que omita la varianza municipal la eleva del 0,85 al 0,86 %, y deja las
dos cifras calculadas sobre bases distintas. Esa comprobación existe para que eso no vuelva a
ocurrir.

## 5. Veredicto

El resumen final reduce el cuadro anterior a las tres cantidades que deciden si la versión es
entregable.

In [ ]:
conteo = resultados["resultado"].value_counts()
coinciden   = int(conteo.get("COINCIDE", 0))
discrepan   = int(conteo.get("DISCREPA", 0))
no_evaluadas = int(conteo.get("NO EVALUADA", 0))

print("VEREDICTO\n" + "=" * 78)
print(f"  coinciden ....... {coinciden:>3} de {len(resultados)}")
print(f"  discrepan ....... {discrepan:>3}")
print(f"  no evaluadas .... {no_evaluadas:>3}")
print("=" * 78)

if discrepan:
    print("\n  HAY DISCREPANCIAS. No dar la versión por entregable hasta resolverlas:\n")
    for _, f in resultados[resultados["resultado"] == "DISCREPA"].iterrows():
        print(f"    {f['id']}  {f['cifra']}")
        print(f"        {f['seccion']}: publicado {f['publicado']:,.4f}, "
              f"obtenido {f['obtenido']:,.4f}, diferencia {f['diferencia']:+,.4f}")
elif no_evaluadas:
    print("\n  Sin discrepancias, pero quedan cifras sin verificar por falta de insumos.")
else:
    print("\n  Todas las cifras comprobadas se reproducen a partir de los datos.")